# Import

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import lightgbm as lgb
import geopandas as gpd

from shapely.geometry import Point
from scipy.optimize import minimize
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from math import radians, sin, cos, sqrt, atan2



# Load Dataset and Cleaning

parse data test supaya kolomnya jadi datetime/nama_pos/id dari yang awalnya cmn id doang

In [2]:
# 1. Load raw files
train = pd.read_csv('../data/train.csv', parse_dates=['datetime'])
test_raw = pd.read_csv('../data/test.csv')
env = pd.read_csv('../data/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
koor = pd.read_csv('../data/data_pendukung/koordinat_pos.csv')

# 2. Parse kolom 'id' di test.csv -> jadi 'datetime' dan 'nama_pos'
# format id: "2025-09-19 06:00:00 - Arjowinangun - Pacitan"
# split hanya di kemunculan PERTAMA " - " (karena nama pos bisa mengandung " - " juga)
split_result = test_raw['id'].str.split(' - ', n=1, expand=True)
test = pd.DataFrame({
    'datetime': pd.to_datetime(split_result[0]),
    'nama_pos': split_result[1],
    'id': test_raw['id']   # simpan buat submission nanti
})

# test.to_csv("test_tes.csv", index=False)

print(test['nama_pos'].nunique())  # pastikan tetap 30
print(test.head())

30
             datetime                nama_pos  \
0 2025-09-19 06:00:00  Arjowinangun - Pacitan   
1 2025-09-19 12:00:00  Arjowinangun - Pacitan   
2 2025-09-19 18:00:00  Arjowinangun - Pacitan   
3 2025-09-20 06:00:00  Arjowinangun - Pacitan   
4 2025-09-20 12:00:00  Arjowinangun - Pacitan   

                                             id  
0  2025-09-19 06:00:00 - Arjowinangun - Pacitan  
1  2025-09-19 12:00:00 - Arjowinangun - Pacitan  
2  2025-09-19 18:00:00 - Arjowinangun - Pacitan  
3  2025-09-20 06:00:00 - Arjowinangun - Pacitan  
4  2025-09-20 12:00:00 - Arjowinangun - Pacitan  


gabung semua dataset train dan test untuk persiapan aggregation kolom dengan data lingkungan

In [3]:
train['is_train'] = 1
test['is_train'] = 0
test['tma_mdpl'] = np.nan   # kosongkan target, akan diisi model nanti

full = pd.concat([
    train[['datetime', 'nama_pos', 'tma_mdpl', 'is_train']],
    test[['datetime', 'nama_pos', 'tma_mdpl', 'is_train', 'id']]
], ignore_index=True, sort=False)

full = full.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
print(full.shape)   # harus = 84396 + 21780
full.head()

# full.to_csv("full_data.csv",index=False)

(106176, 5)


,datetime,nama_pos,tma_mdpl,is_train,id
0,2023-01-01 06:00:00,Arjowinangun - Pacitan,1.30,1,NaN
1,2023-01-01 12:00:00,Arjowinangun - Pacitan,1.20,1,NaN
2,2023-01-01 18:00:00,Arjowinangun - Pacitan,1.50,1,NaN
3,2023-01-02 06:00:00,Arjowinangun - Pacitan,1.45,1,NaN
4,2023-01-02 12:00:00,Arjowinangun - Pacitan,1.25,1,NaN


gabungkan data lingkungan ke data fullnya (+ cleaned dari -999 di solar radiation)

In [4]:
full = pd.read_csv('../data/full_data.csv', parse_dates=['datetime'])
env = pd.read_csv('../data/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])

# === LANGKAH BARU: bersihkan sentinel -999 SEBELUM agregasi ===
numeric_cols = env.select_dtypes(include='number').columns
for col in numeric_cols:
    n_sentinel = (env[col] == -999).sum()
    if n_sentinel > 0:
        print(f'{col}: {n_sentinel} baris sentinel -999 -> diubah jadi NaN')
        env[col] = env[col].replace(-999, np.nan)

# === LANGKAH LAMA: assign window_end (tidak berubah) ===
hour = env['datetime'].dt.hour
date = env['datetime'].dt.normalize()
conditions = [hour < 6, hour < 12, hour < 18]
choices = [date + pd.Timedelta(hours=6), date + pd.Timedelta(hours=12), date + pd.Timedelta(hours=18)]
default = date + pd.Timedelta(days=1, hours=6)
env['window_end'] = np.select(conditions, choices, default=default)

# === LANGKAH LAMA: agregasi (tidak berubah, tapi hasilnya sekarang aman krn NaN otomatis di-skip oleh sum/mean) ===
agg_dict = {
    'rainfall_mm': 'sum', 'rainfall_openmeteo_mm': 'sum', 'rainfall_max_24h_mm': 'max',
    'humidity_pct': 'mean', 'wind_direction_deg': 'mean', 'dew_point_c': 'mean',
    'cloud_cover_pct': 'mean', 'temperature_c': 'mean', 'wind_speed_kmh': 'mean',
    'solar_radiation_mj_m2': 'sum', 'soil_moisture_0_7cm': 'mean', 'soil_moisture_7_28cm': 'mean',
    'soil_moisture_28_100cm': 'mean', 'soil_moisture_100_255cm': 'mean',
    'surface_pressure_hpa': 'mean', 'pressure_msl_hpa': 'mean', 'built_surface_m2': 'first',
    'landcover_class': 'first', 'landcover_name': 'first', 'rmm1': 'mean', 'rmm2': 'mean',
    'mjo_phase': 'first', 'mjo_amplitude': 'mean', 'mjo_active': 'first', 'nino_34': 'mean',
}
env_agg = env.groupby(['nama_pos', 'window_end']).agg(agg_dict).reset_index()

sum_cols = ['rainfall_mm', 'rainfall_openmeteo_mm', 'solar_radiation_mj_m2']

for col in sum_cols:
    all_nan_mask = env.groupby(['nama_pos', 'window_end'])[col].apply(lambda x: x.isna().all())
    n_affected = all_nan_mask.sum()
    print(f'{col}: {n_affected} window full-NaN -> override balik jadi NaN')
    # all_nan_mask index-nya sama (nama_pos, window_end) dgn env_agg, jadi bisa langsung dipakai buat masking
    env_agg.loc[all_nan_mask.values, col] = np.nan
    
env_agg = env_agg.rename(columns={'window_end': 'datetime'})

full = full.merge(env_agg, on=['nama_pos', 'datetime'], how='left')

# === CEK HASIL ===
print(full['solar_radiation_mj_m2'].describe())   # min sekarang harusnya wajar, tidak lagi -11988
print(full[full['datetime'] >= '2025-12-31'][['solar_radiation_mj_m2','soil_moisture_0_7cm','nino_34']].isna().sum())

# full.to_csv("../data/full_data_merged.csv",index=False)

solar_radiation_mj_m2: 100080 baris sentinel -999 -> diubah jadi NaN
rainfall_mm: 0 window full-NaN -> override balik jadi NaN
rainfall_openmeteo_mm: 0 window full-NaN -> override balik jadi NaN
solar_radiation_mj_m2: 12510 window full-NaN -> override balik jadi NaN
count    93696.000000
mean      1719.582734
std       1338.502936
min          0.000000
25%         14.120000
50%       2084.360000
75%       2836.980000
max       4387.940000
Name: solar_radiation_mj_m2, dtype: float64
solar_radiation_mj_m2    12480
soil_moisture_0_7cm         60
nino_34                   1590
dtype: int64


In [5]:
print("""
    note: pada tgl 4 feb - 28 feb 2025 itu gaada datanya jadi abis 3 feb langsung ke 1 maret 2025, ntar bakal di bring up
    di preprocessing

""")


    note: pada tgl 4 feb - 28 feb 2025 itu gaada datanya jadi abis 3 feb langsung ke 1 maret 2025, ntar bakal di bring up
    di preprocessing




## feature engineering

### check dataset gabungan

In [6]:
full.head()

,datetime,nama_pos,tma_mdpl,is_train,id,rainfall_mm,rainfall_openmeteo_mm,rainfall_max_24h_mm,humidity_pct,wind_direction_deg,...,pressure_msl_hpa,built_surface_m2,landcover_class,landcover_name,rmm1,rmm2,mjo_phase,mjo_amplitude,mjo_active,nino_34
0,2023-01-01 06:00:00,Arjowinangun - Pacitan,1.30,1,NaN,0.0,0.0,0.0,86.000000,348.333333,...,1009.066667,2419.0,10,Tree cover,-0.092299,1.277599,7.0,1.280929,1.0,-0.72
1,2023-01-01 12:00:00,Arjowinangun - Pacitan,1.20,1,NaN,0.3,0.3,0.2,79.166667,327.666667,...,1010.800000,2419.0,10,Tree cover,-0.092299,1.277599,7.0,1.280929,1.0,-0.72
2,2023-01-01 18:00:00,Arjowinangun - Pacitan,1.50,1,NaN,3.6,3.6,1.7,81.666667,296.333333,...,1008.983333,2419.0,10,Tree cover,-0.092299,1.277599,7.0,1.280929,1.0,-0.72
3,2023-01-02 06:00:00,Arjowinangun - Pacitan,1.45,1,NaN,1.0,1.0,1.7,90.250000,316.166667,...,1010.000000,2419.0,10,Tree cover,-0.311643,1.335574,7.0,1.386106,1.0,-0.72
4,2023-01-02 12:00:00,Arjowinangun - Pacitan,1.25,1,NaN,0.6,0.6,1.7,79.500000,298.166667,...,1010.766667,2419.0,10,Tree cover,-0.530987,1.393549,7.0,1.491283,1.0,-0.72


### dropping Columns

In [7]:


full.drop([
    "rainfall_openmeteo_mm", # korelasi = 1
], axis=1, inplace=True)

In [8]:
river = gpd.read_file("../data/data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp")
river = river.to_crs(4326)

print(river.columns)

Index(['HYRIV_ID', 'NEXT_DOWN', 'MAIN_RIV', 'LENGTH_KM', 'DIST_DN_KM',
       'DIST_UP_KM', 'CATCH_SKM', 'UPLAND_SKM', 'ENDORHEIC', 'DIS_AV_CMS',
       'ORD_STRA', 'ORD_CLAS', 'ORD_FLOW', 'HYBAS_L12', 'geometry'],
      dtype='object')


In [9]:
full = pd.read_csv('../data/full_data_merged.csv', parse_dates=['datetime'])
koor = pd.read_csv('../data/data_pendukung/koordinat_pos.csv')

def detect_and_fix_glitches(df, window=4, ratio_threshold=4, zscore_threshold=10, min_abs_jump=1.0):
    df = df.sort_values(['nama_pos','datetime'])  # TANPA reset_index (biar index tetap nyambung ke full)
    glitch_indices = []
    for pos, g in df.groupby('nama_pos'):
        val = g['tma_mdpl'].values
        idx_list = g.index.tolist()
        local_std = np.std(val)
        for i in range(window, len(val)-window):
            neighborhood = np.concatenate([val[i-window:i], val[i+1:i+1+window]])
            local_median = np.median(neighborhood)
            cur_v = val[i]
            jump = abs(cur_v - local_median)
            if jump < min_abs_jump:
                continue
            ratio = max(cur_v/local_median, local_median/cur_v) if cur_v>0 and local_median>0 else 999
            is_extreme = (ratio > ratio_threshold) or (local_std > 0 and (jump/local_std) > zscore_threshold)
            if is_extreme:
                glitch_indices.append(idx_list[i])
    print(f'Ditemukan {len(glitch_indices)} titik glitch')
    df.loc[glitch_indices, 'tma_mdpl'] = np.nan
    df['tma_mdpl'] = df.groupby('nama_pos')['tma_mdpl'].transform(lambda x: x.interpolate(method='linear'))
    return df, glitch_indices

train_part = full[full['is_train']==1].copy()
train_part_fixed, glitch_idx = detect_and_fix_glitches(train_part)
full.loc[train_part_fixed.index, 'tma_mdpl'] = train_part_fixed['tma_mdpl']

Ditemukan 24 titik glitch


### adding features

In [10]:
# full = pd.read_csv('../data/full_data_merged.csv', parse_dates=['datetime'])
# koor = pd.read_csv('../data/data_pendukung/koordinat_pos.csv')

full.drop(['rainfall_openmeteo_mm'], axis=1, inplace=True)
full = full.sort_values(['nama_pos','datetime']).reset_index(drop=True)

# ===== reindex grid 6-jaman (msh perlu, krn rain/soil_moisture lag-roll butuh alignment waktu) =====
grids = []
for pos, g in full.groupby('nama_pos'):
    full_range = pd.date_range(g['datetime'].min(), g['datetime'].max(), freq='6h')
    full_range = full_range[full_range.hour.isin([6,12,18])]
    grids.append(pd.DataFrame({'nama_pos': pos, 'datetime': full_range}))
grid = pd.concat(grids, ignore_index=True)
full = grid.merge(full, on=['nama_pos','datetime'], how='left')
full['is_phantom'] = full['is_train'].isna().astype(int)
full = full.sort_values(['nama_pos','datetime']).reset_index(drop=True)

# ===== FITUR WAKTU (lebih lengkap) =====
full['hour'] = full['datetime'].dt.hour
full['day'] = full['datetime'].dt.day
full['month'] = full['datetime'].dt.month
full['week'] = full['datetime'].dt.isocalendar().week.astype(int)
full['quarter'] = full['datetime'].dt.quarter
full['dayofyear'] = full['datetime'].dt.dayofyear
full['dayofweek'] = full['datetime'].dt.dayofweek
full['is_weekend'] = full['dayofweek'].isin([5,6]).astype(int)
full['hour_sin'] = np.sin(2*np.pi*full['hour']/24)
full['hour_cos'] = np.cos(2*np.pi*full['hour']/24)
full['month_sin'] = np.sin(2*np.pi*full['month']/12)
full['month_cos'] = np.cos(2*np.pi*full['month']/12)
full['dayofyear_sin'] = np.sin(2*np.pi*full['dayofyear']/365)
full['dayofyear_cos'] = np.cos(2*np.pi*full['dayofyear']/365)

# ===== FITUR CUACA: lag & rolling (SEMUA berbasis cuaca, BUKAN target -> gak butuh recursion) =====
g = full.groupby('nama_pos')

def rollsum(col, n): return g[col].transform(lambda x: x.shift(1).rolling(n).sum())
def rollmean(col, n): return g[col].transform(lambda x: x.shift(1).rolling(n).mean())

full['rainfall_mm_lag_1'] = g['rainfall_mm'].shift(1)
full['rainfall_mm_lag_3'] = g['rainfall_mm'].shift(3)
full['rainfall_mm_lag_6'] = g['rainfall_mm'].shift(6)
full['humidity_lag_1'] = g['humidity_pct'].shift(1)
full['soil_moisture_0_7cm_lag_1'] = g['soil_moisture_0_7cm'].shift(1)
full['soil_moisture_7_28cm_lag_1'] = g['soil_moisture_7_28cm'].shift(1)
full['soil_moisture_28_100cm_lag_1'] = g['soil_moisture_28_100cm'].shift(1)
full['soil_moisture_100_255cm_lag_1'] = g['soil_moisture_100_255cm'].shift(1)

full['rainfall_mm_roll_sum_6'] = rollsum('rainfall_mm', 6)
full['rainfall_mm_roll_sum_12'] = rollsum('rainfall_mm', 12)
full['soil_moisture_0_7cm_roll_mean_6'] = rollmean('soil_moisture_0_7cm', 6)
full['soil_moisture_7_28cm_roll_mean_6'] = rollmean('soil_moisture_7_28cm', 6)
full['soil_moisture_28_100cm_roll_mean_6'] = rollmean('soil_moisture_28_100cm', 6)
full['soil_moisture_100_255cm_roll_mean_6'] = rollmean('soil_moisture_100_255cm', 6)

# akumulasi hujan multi-window (48h=8 periode, 96h=16, 144h=24, 168h=28, 336h=56 -- satuan 6 jam-an)
full['rain_cumsum_48h']  = rollsum('rainfall_mm', 8)
full['rain_cumsum_96h']  = rollsum('rainfall_mm', 16)
full['rain_cumsum_144h'] = rollsum('rainfall_mm', 24)
full['rain_cumsum_168h'] = rollsum('rainfall_mm', 28)
full['rain_cumsum_336h'] = rollsum('rainfall_mm', 56)

# indeks iklim: rolling mean jangka panjang (120 & 240 periode 6-jaman = ~30 & ~60 hari)
for col in ['rmm1','rmm2','mjo_amplitude','nino_34']:
    full[f'{col}_roll_mean_120'] = rollmean(col, 120)
    full[f'{col}_roll_mean_240'] = rollmean(col, 240)

full['delta_rain_1'] = g['rainfall_mm'].diff(1)
full['delta_pressure_1'] = g['surface_pressure_hpa'].diff(1)

full['wind_dir_sin'] = np.sin(np.radians(full['wind_direction_deg']))
full['wind_dir_cos'] = np.cos(np.radians(full['wind_direction_deg']))

# ===== buang phantom, gabung lokasi =====
full = full[full['is_phantom']==0].drop(columns=['is_phantom']).reset_index(drop=True)
full = full.merge(koor, on='nama_pos', how='left')

# ===== FITUR STASIUN STATIS (KUNCI: gantikan tma_lag tanpa butuh recursion) =====
train_mask = full['is_train']==1
station_stats = full[train_mask].groupby('nama_pos')['tma_mdpl'].agg(
    station_mean_tma='mean', station_min_tma='min', station_max_tma='max').reset_index()
full = full.merge(station_stats, on='nama_pos', how='left')

# ===== nearest_station_dist (haversine, dari koordinat) =====
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a), sqrt(1-a))

pos_coords = koor.set_index('nama_pos')[['latitude','longitude']]
nearest_dist = {p1: min(haversine(*pos_coords.loc[p1], *pos_coords.loc[p2])
                          for p2 in pos_coords.index if p2 != p1)
                 for p1 in pos_coords.index}
full['nearest_station_dist'] = full['nama_pos'].map(nearest_dist)

# ===== encoding kategorikal =====
from sklearn.preprocessing import LabelEncoder
for col in ['nama_pos','landcover_name','mjo_phase','mjo_active']:
    le = LabelEncoder()
    full[col+'_enc'] = le.fit_transform(full[col].astype(str))

train = full[full['is_train']==1].copy()
test = full[full['is_train']==0].copy()
non_feature = ['datetime','nama_pos','is_train','id','tma_mdpl','landcover_name','mjo_phase','mjo_active']
feature_cols = [c for c in train.columns if c not in non_feature]
print(train.shape, test.shape, len(feature_cols), 'fitur')

(84396, 84) (21780, 84) 76 fitur


In [11]:
# # =====================================================
# # HYDRORIVERS MERGE
# # =====================================================

# # pastikan CRS sama
# # river = river.to_crs(4326)

# # koordinat station
# station_gdf = gpd.GeoDataFrame(
#     koor.copy(),
#     geometry=gpd.points_from_xy(
#         koor.longitude,
#         koor.latitude
#     ),
#     crs="EPSG:4326"
# )

# # nearest river
# nearest = gpd.sjoin_nearest(
#     station_gdf,
#     river[
#         [
#             "HYRIV_ID",
#             "LENGTH_KM",
#             "DIST_DN_KM",
#             "DIST_UP_KM",
#             "CATCH_SKM",
#             "UPLAND_SKM",
#             "DIS_AV_CMS",
#             "ORD_STRA",
#             "ORD_CLAS",
#             "ORD_FLOW",
#             "ENDORHEIC",
#             "geometry"
#         ]
#     ],
#     how="left",
#     distance_col="river_distance"
# )

# river_feature = nearest.drop(columns="geometry")

# # merge ke train
# full = full.merge(
#     river_feature,
#     on="nama_pos",
#     how="left"
# )

# print("HydroRIVERS merged.")

In [12]:
# # =====================================================
# # HYDRORIVERS FEATURE ENGINEERING
# # =====================================================

# for df in [full]:

#     df["river_distance_log"] = np.log1p(df["river_distance"])

#     df["river_length_log"] = np.log1p(df["LENGTH_KM"])

#     df["catchment_log"] = np.log1p(df["CATCH_SKM"])

#     df["upstream_log"] = np.log1p(df["UPLAND_SKM"])

#     df["discharge_log"] = np.log1p(df["DIS_AV_CMS"])

#     # rasio catchment
#     df["catchment_ratio"] = (
#         df["UPLAND_SKM"] /
#         (df["CATCH_SKM"] + 1)
#     )

#     # gradient sungai
#     df["river_gradient"] = (
#         df["DIST_UP_KM"] /
#         (df["DIST_DN_KM"] + 1)
#     )

#     # river density
#     df["river_density"] = (
#         df["ORD_FLOW"] /
#         (df["river_distance"] + 0.001)
#     )

#     # discharge density
#     df["discharge_density"] = (
#         df["DIS_AV_CMS"] /
#         (df["river_distance"] + 0.001)
#     )

#     # rainfall interaction
#     df["rain_x_discharge"] = (
#         df["rainfall_mm"] *
#         df["DIS_AV_CMS"]
#     )

#     df["rain_x_upstream"] = (
#         df["rainfall_mm"] *
#         df["UPLAND_SKM"]
#     )

#     df["rain_x_order"] = (
#         df["rainfall_mm"] *
#         df["ORD_FLOW"]
#     )

#     # soil interaction
#     df["soil_x_upstream"] = (
#         df["soil_moisture_0_7cm"] *
#         df["UPLAND_SKM"]
#     )

#     # humidity interaction
#     df["humidity_x_discharge"] = (
#         df["humidity_pct"] *
#         df["DIS_AV_CMS"]
#     )

#     # pressure interaction
#     df["pressure_x_discharge"] = (
#         df["surface_pressure_hpa"] *
#         df["DIS_AV_CMS"]
#     )

#     # wind interaction
#     df["wind_x_order"] = (
#         df["wind_speed_kmh"] *
#         df["ORD_FLOW"]
#     )

#     # rainfall per catchment
#     df["rain_per_catchment"] = (
#         df["rainfall_mm"] /
#         (df["CATCH_SKM"] + 1)
#     )

#     # rainfall per upstream
#     df["rain_per_upstream"] = (
#         df["rainfall_mm"] /
#         (df["UPLAND_SKM"] + 1)
#     )

# print("HydroRIVERS features created.")

In [13]:
non_feature = [
    'datetime',
    'nama_pos',
    'is_train',
    'id',
    'tma_mdpl',
    'landcover_name',
    'mjo_phase',
    'mjo_active'
]

feature_cols = [c for c in train.columns if c not in non_feature]

print(len(feature_cols))

76


In [14]:
train_only = full[full["is_train"] == 1].copy()

station_mean = (
    train_only
    .groupby("nama_pos")["tma_mdpl"]
    .mean()
)

global_mean = train_only["tma_mdpl"].mean()

full["nama_pos_te"] = (
    full["nama_pos"]
    .map(station_mean)
    .fillna(global_mean)
)

## finalize before split train and val

balikin train ke train dan test ke test dari full, remove non feature columns

In [15]:
train = full[full['is_train']==1].copy() # balikin train ke train
test = full[full['is_train']==0].copy() # balikin test ke test

non_feature = ['datetime','nama_pos','is_train','id','tma_mdpl','landcover_name','mjo_phase','mjo_active'] # remove non feature column
feature_cols = [c for c in train.columns if c not in non_feature] 
print(train.shape, test.shape, len(feature_cols), 'fitur')

(84396, 85) (21780, 85) 77 fitur


In [16]:
train.head()

,nama_pos,datetime,tma_mdpl,is_train,id,rainfall_mm,rainfall_max_24h_mm,humidity_pct,wind_direction_deg,dew_point_c,...,longitude,station_mean_tma,station_min_tma,station_max_tma,nearest_station_dist,nama_pos_enc,landcover_name_enc,mjo_phase_enc,mjo_active_enc,nama_pos_te
0,Arjowinangun - Pacitan,2023-01-01 06:00:00,1.30,1.0,NaN,0.0,0.0,86.000000,348.333333,22.466667,...,111.114314,1.116478,0.347639,4.85,7.188337,0,3,6,1,1.116478
1,Arjowinangun - Pacitan,2023-01-01 12:00:00,1.20,1.0,NaN,0.3,0.2,79.166667,327.666667,23.550000,...,111.114314,1.116478,0.347639,4.85,7.188337,0,3,6,1,1.116478
2,Arjowinangun - Pacitan,2023-01-01 18:00:00,1.50,1.0,NaN,3.6,1.7,81.666667,296.333333,24.050000,...,111.114314,1.116478,0.347639,4.85,7.188337,0,3,6,1,1.116478
3,Arjowinangun - Pacitan,2023-01-02 06:00:00,1.45,1.0,NaN,1.0,1.7,90.250000,316.166667,23.316667,...,111.114314,1.116478,0.347639,4.85,7.188337,0,3,6,1,1.116478
4,Arjowinangun - Pacitan,2023-01-02 12:00:00,1.25,1.0,NaN,0.6,1.7,79.500000,298.166667,23.333333,...,111.114314,1.116478,0.347639,4.85,7.188337,0,3,6,1,1.116478


## split data train val

In [17]:
cutoff = pd.Timestamp('2025-03-03')  # ~80% waktu pertama utk train, 20% terakhir utk val

train_set = train[train['datetime'] < cutoff].copy()
val_set = train[train['datetime'] >= cutoff].copy()

print(train_set.shape, val_set.shape)
print('train_set:', train_set['datetime'].min(), '-', train_set['datetime'].max())
print('val_set:', val_set['datetime'].min(), '-', val_set['datetime'].max())
# pastikan semua 30 pos ada di kedua set
print('pos di train_set:', train_set['nama_pos'].nunique(), '| pos di val_set:', val_set['nama_pos'].nunique())

(66441, 85) (17955, 85)
train_set: 2023-01-01 06:00:00 - 2025-03-02 18:00:00
val_set: 2025-03-03 06:00:00 - 2025-09-18 18:00:00
pos di train_set: 30 | pos di val_set: 30


In [18]:
stats = train_set.groupby('nama_pos')['tma_mdpl'].agg(
    station_mean_tma='mean', station_min_tma='min', station_max_tma='max').reset_index()
te = train_set.groupby('nama_pos')['tma_mdpl'].mean()
gmean = train_set['tma_mdpl'].mean()

drop_cols = ['station_mean_tma','station_min_tma','station_max_tma','nama_pos_te']
train_set = train_set.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
val_set = val_set.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
train_set['nama_pos_te'] = train_set['nama_pos'].map(te).fillna(gmean)
val_set['nama_pos_te'] = val_set['nama_pos'].map(te).fillna(gmean)

# Modelling

In [19]:
# LAGROLL (with short lags removed) - buat variable eksplisit
X_train = train_set[feature_cols]
y_train = train_set['tma_mdpl']
X_val = val_set[feature_cols]
y_val = val_set['tma_mdpl']


In [41]:
non_feature = ['datetime','nama_pos','is_train','id','tma_mdpl','landcover_name','mjo_phase','mjo_active']
feature_cols = [c for c in train.columns if c not in non_feature]

cutoff = pd.Timestamp('2025-03-03')
train_set = train[train['datetime'] < cutoff].copy()
val_set = train[train['datetime'] >= cutoff].copy()

# cutoffs = [
#     ('2024-09-01', '2025-03-03'),
#     ('2024-12-01', '2025-06-01'),
#     ('2025-03-03', '2025-09-18'),
# ]

cutoffs = [
    ('2023-11-01', '2024-05-01'),   # va: Nov'23-Mei'24, penuh musim hujan
    ('2024-11-01', '2025-05-01'),   # va: Nov'24-Mei'25, penuh musim hujan
    ('2024-06-01', '2025-01-01'),   # va: campuran kemarau→hujan
]

xgb_params  = dict(n_estimators=350,  learning_rate=0.04, max_depth=6,
                    subsample=0.7, colsample_bytree=0.8, random_state=42)
lgbm_params = dict(n_estimators=2000, learning_rate=0.04, max_depth=6,
                    subsample=0.8, colsample_bytree=0.7, random_state=42, verbosity=-1)
cat_params  = dict(iterations=2250,  learning_rate=0.04, depth=6,
                    subsample=0.8, colsample_bylevel=0.8, random_state=42, verbose=False)

### XGBoost

In [42]:
xgb = XGBRegressor(**xgb_params)
xgb.fit(train_set[feature_cols], train_set['tma_mdpl'])

pred_val_xgb = xgb.predict(val_set[feature_cols])
rmse_xgb = np.sqrt(mean_squared_error(val_set['tma_mdpl'], pred_val_xgb))
print(f'RMSE XGBoost (single split): {rmse_xgb:.4f}')

RMSE XGBoost (single split): 0.7987


In [61]:
results_xgb = []
for train_end, val_end in cutoffs:
    tr = train[train['datetime'] < train_end]
    va = train[(train['datetime'] >= train_end) & (train['datetime'] < val_end)]

    stats = tr.groupby('nama_pos')['tma_mdpl'].agg(
        station_mean_tma='mean', station_min_tma='min', station_max_tma='max').reset_index()
    te = tr.groupby('nama_pos')['tma_mdpl'].mean()
    gmean = tr['tma_mdpl'].mean()
    gmin = tr['tma_mdpl'].min()
    gmax = tr['tma_mdpl'].max()

    drop_cols = ['station_mean_tma','station_min_tma','station_max_tma','nama_pos_te']
    tr = tr.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
    va = va.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')

    tr['nama_pos_te'] = tr['nama_pos'].map(te).fillna(gmean)
    va['nama_pos_te'] = va['nama_pos'].map(te).fillna(gmean)

    # TAMBAHAN INI — biar gak NaN kalau ada stasiun baru yang belum ada histori:
    for col, fallback in [('station_mean_tma', gmean), ('station_min_tma', gmin), ('station_max_tma', gmax)]:
        tr[col] = tr[col].fillna(fallback)
        va[col] = va[col].fillna(fallback)
    model = XGBRegressor(**xgb_params)   # ganti model sesuai cell
    model.fit(tr[feature_cols], tr['tma_mdpl'])
    pred = model.predict(va[feature_cols])
    rmse = np.sqrt(mean_squared_error(va['tma_mdpl'], pred))
    results_xgb.append(rmse)
    print(f'Fold RMSE: {rmse:.4f}')
print(f'RMSE CV rata-rata: {np.mean(results_xgb):.4f} +/- {np.std(results_xgb):.4f}')

Fold RMSE: 1.4552
Fold RMSE: 1.1582
Fold RMSE: 5.1290
RMSE CV rata-rata: 2.5808 +/- 1.8059


In [44]:
print(f'RMSE CV rata-rata: {np.mean(results_xgb):.4f} +/- {np.std(results_xgb):.4f}')

RMSE CV rata-rata: 6.6393 +/- 7.5424


## LGBM

In [45]:
lgbm = LGBMRegressor(**lgbm_params)
lgbm.fit(train_set[feature_cols], train_set['tma_mdpl'])

pred_val_lgbm = lgbm.predict(val_set[feature_cols])
rmse_lgbm = np.sqrt(mean_squared_error(val_set['tma_mdpl'], pred_val_lgbm))
print(f'RMSE LightGBM (single split): {rmse_lgbm:.4f}')

RMSE LightGBM (single split): 0.8064


In [46]:
results_lgbm = []
for train_end, val_end in cutoffs:
    tr = train[train['datetime'] < train_end]
    va = train[(train['datetime'] >= train_end) & (train['datetime'] < val_end)]

    stats = tr.groupby('nama_pos')['tma_mdpl'].agg(
        station_mean_tma='mean', station_min_tma='min', station_max_tma='max').reset_index()
    te = tr.groupby('nama_pos')['tma_mdpl'].mean()
    gmean = tr['tma_mdpl'].mean()

    drop_cols = ['station_mean_tma','station_min_tma','station_max_tma','nama_pos_te']
    tr = tr.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
    va = va.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
    tr['nama_pos_te'] = tr['nama_pos'].map(te).fillna(gmean)
    va['nama_pos_te'] = va['nama_pos'].map(te).fillna(gmean)

    model = LGBMRegressor(**lgbm_params)   # ganti model sesuai cell
    model.fit(tr[feature_cols], tr['tma_mdpl'])
    pred = model.predict(va[feature_cols])
    rmse = np.sqrt(mean_squared_error(va['tma_mdpl'], pred))
    results_lgbm.append(rmse)
    print(f'Fold RMSE: {rmse:.4f}')
print(f'RMSE CV rata-rata: {np.mean(results_lgbm):.4f} +/- {np.std(results_lgbm):.4f}')

Fold RMSE: 3.6134
Fold RMSE: 1.2425
Fold RMSE: 2.9842
RMSE CV rata-rata: 2.6134 +/- 1.0028


In [47]:
print(f'RMSE CV rata-rata: {np.mean(results_lgbm):.4f} +/- {np.std(results_lgbm):.4f}')

RMSE CV rata-rata: 2.6134 +/- 1.0028


## CatBoost

In [48]:
cat = CatBoostRegressor(**cat_params)
cat.fit(train_set[feature_cols], train_set['tma_mdpl'])

pred_val_cat = cat.predict(val_set[feature_cols])
rmse_cat = np.sqrt(mean_squared_error(val_set['tma_mdpl'], pred_val_cat))
print(f'RMSE CatBoost (single split): {rmse_cat:.4f}')

RMSE CatBoost (single split): 0.7309


In [49]:
results_cat = []
for train_end, val_end in cutoffs:
    tr = train[train['datetime'] < train_end]
    va = train[(train['datetime'] >= train_end) & (train['datetime'] < val_end)]

    stats = tr.groupby('nama_pos')['tma_mdpl'].agg(
        station_mean_tma='mean', station_min_tma='min', station_max_tma='max').reset_index()
    te = tr.groupby('nama_pos')['tma_mdpl'].mean()
    gmean = tr['tma_mdpl'].mean()

    drop_cols = ['station_mean_tma','station_min_tma','station_max_tma','nama_pos_te']
    tr = tr.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
    va = va.drop(columns=drop_cols).merge(stats, on='nama_pos', how='left')
    tr['nama_pos_te'] = tr['nama_pos'].map(te).fillna(gmean)
    va['nama_pos_te'] = va['nama_pos'].map(te).fillna(gmean)

    model = CatBoostRegressor(**cat_params)   # ganti model sesuai cell
    model.fit(tr[feature_cols], tr['tma_mdpl'])
    pred = model.predict(va[feature_cols])
    rmse = np.sqrt(mean_squared_error(va['tma_mdpl'], pred))
    results_cat.append(rmse)
    print(f'Fold RMSE: {rmse:.4f}')
print(f'RMSE CV rata-rata: {np.mean(results_cat):.4f} +/- {np.std(results_cat):.4f}')

Fold RMSE: 1.3270
Fold RMSE: 1.1412
Fold RMSE: 1.4839
RMSE CV rata-rata: 1.3174 +/- 0.1400


In [ ]:
for train_end, val_end in cutoffs:
    va = train[(train['datetime'] >= train_end) & (train['datetime'] < val_end)]
    pct_hujan = va['datetime'].dt.month.isin([11,12,1,2,3]).mean()*100
    print(f'{train_end} -> {val_end}: {len(va)} baris, {pct_hujan:.1f}% musim hujan')

In [50]:
print(f'RMSE CV rata-rata: {np.mean(results_cat):.4f} +/- {np.std(results_cat):.4f}')

RMSE CV rata-rata: 1.3174 +/- 0.1400


## Perbandingan XGB vs LGBM vs CatBoost

In [51]:
comparison = pd.DataFrame({
    'model': ['XGBoost', 'LightGBM', 'CatBoost'],
    'RMSE_single_split': [rmse_xgb, rmse_lgbm, rmse_cat],
    'RMSE_CV_mean': [np.mean(results_xgb), np.mean(results_lgbm), np.mean(results_cat)],
    'RMSE_CV_std': [np.std(results_xgb), np.std(results_lgbm), np.std(results_cat)],
})
comparison['RMSE_CV_mean_pm_std'] = comparison.apply(
    lambda r: f"{r['RMSE_CV_mean']:.4f} ± {r['RMSE_CV_std']:.4f}", axis=1
)
comparison_display = comparison[['model','RMSE_single_split','RMSE_CV_mean_pm_std']].sort_values('RMSE_single_split')
print(comparison_display.to_string(index=False))
comparison_display

   model  RMSE_single_split RMSE_CV_mean_pm_std
CatBoost           0.730932     1.3174 ± 0.1400
 XGBoost           0.798735     6.6393 ± 7.5424
LightGBM           0.806385     2.6134 ± 1.0028


,model,RMSE_single_split,RMSE_CV_mean_pm_std
2,CatBoost,0.730932,1.3174 ± 0.1400
0,XGBoost,0.798735,6.6393 ± 7.5424
1,LightGBM,0.806385,2.6134 ± 1.0028


## Voting Ensemble

In [52]:
# # ===== 1. cari bobot optimal & evaluasi di val_set dulu =====
# def neg_rmse_weighted(weights, preds_list, y_true):
#     w = np.array(weights) / np.sum(weights)
#     blend = sum(w_i * p for w_i, p in zip(w, preds_list))
#     return np.sqrt(mean_squared_error(y_true, blend))

# preds_list_val = [pred_val_cat, pred_val_xgb, pred_val_lgbm]
# res = minimize(neg_rmse_weighted, x0=[1,1,1], args=(preds_list_val, val_set['tma_mdpl'].values),
#                 method='Nelder-Mead', bounds=[(0,None)]*3)
# best_weights = np.array(res.x) / np.sum(res.x)
# print('Bobot optimal (cat, xgb, lgbm):', np.round(best_weights,3))

# pred_val_ensemble = (best_weights[0]*pred_val_cat +
#                       best_weights[1]*pred_val_xgb +
#                       best_weights[2]*pred_val_lgbm)

# rmse_val_ensemble = np.sqrt(mean_squared_error(val_set['tma_mdpl'], pred_val_ensemble))
# print(f'RMSE CatBoost saja   : {np.sqrt(mean_squared_error(val_set["tma_mdpl"], pred_val_cat)):.4f}')
# print(f'RMSE XGBoost saja    : {np.sqrt(mean_squared_error(val_set["tma_mdpl"], pred_val_xgb)):.4f}')
# print(f'RMSE LightGBM saja   : {np.sqrt(mean_squared_error(val_set["tma_mdpl"], pred_val_lgbm)):.4f}')
# print(f'RMSE Ensemble (bobot): {rmse_val_ensemble:.4f}')

In [53]:
# cutoffs = [
#     ('2024-09-01', '2025-03-03'),
#     ('2024-12-01', '2025-06-01'),
#     ('2025-03-03', '2025-09-18'),
# ]

# results_ensemble = []
# for train_end, val_end in cutoffs:
#     tr = train[train['datetime'] < train_end]
#     va = train[(train['datetime'] >= train_end) & (train['datetime'] < val_end)]

#     m_cat = CatBoostRegressor(**cat_params); m_cat.fit(tr[feature_cols], tr['tma_mdpl'])
#     m_xgb = XGBRegressor(**xgb_params); m_xgb.fit(tr[feature_cols], tr['tma_mdpl'])
#     m_lgbm = LGBMRegressor(**lgbm_params); m_lgbm.fit(tr[feature_cols], tr['tma_mdpl'])

#     p_cat = m_cat.predict(va[feature_cols])
#     p_xgb = m_xgb.predict(va[feature_cols])
#     p_lgbm = m_lgbm.predict(va[feature_cols])

#     p_ensemble = best_weights[0]*p_cat + best_weights[1]*p_xgb + best_weights[2]*p_lgbm
#     rmse = np.sqrt(mean_squared_error(va['tma_mdpl'], p_ensemble))
#     results_ensemble.append(rmse)
#     print(f'Fold RMSE ensemble: {rmse:.4f}')

# print(f'RMSE ensemble CV rata-rata: {np.mean(results_ensemble):.4f} +/- {np.std(results_ensemble):.4f}')

In [54]:
val_set = val_set.copy()
val_set['pred'] = xgb.predict(val_set[feature_cols])  # ganti 'cat' sesuai model kamu

# RMSE per pos + NRMSE (dinormalisasi thd rentang tma pos itu sendiri, biar ADIL lintas skala)
per_pos = val_set.groupby('nama_pos').apply(
    lambda d: pd.Series({
        'rmse': np.sqrt(mean_squared_error(d['tma_mdpl'], d['pred'])),
        'tma_range': train_set[train_set['nama_pos']==d.name]['tma_mdpl'].max() - train_set[train_set['nama_pos']==d.name]['tma_mdpl'].min(),
        'n_baris': len(d),
    })
).reset_index()
per_pos['nrmse_pct'] = (per_pos['rmse'] / per_pos['tma_range']) * 100
per_pos = per_pos.sort_values('nrmse_pct', ascending=False)

print('=== 10 pos TERBURUK (NRMSE %, adil lintas skala) ===')
print(per_pos.head(10).to_string(index=False))
print()
print('=== 10 pos TERBAIK ===')
print(per_pos.tail(10).to_string(index=False))

=== 10 pos TERBURUK (NRMSE %, adil lintas skala) ===
                nama_pos     rmse  tma_range  n_baris  nrmse_pct
       Boboh Kali Lamong 1.344880   5.500000    600.0  24.452368
              Gunungsari 0.793875   4.682803    600.0  16.952987
       Floodway Bridge C 1.003799   5.924736    600.0  16.942503
            Wonogiri Dam 1.944217  11.710459    600.0  16.602397
             Bengkelolor 0.954109   5.934234    600.0  16.078039
            Karanggeneng 0.643338   5.080000    600.0  12.664136
                   Babat 0.619553   5.250000    586.0  11.801019
                    Cepu 0.897172   7.924874    598.0  11.320968
              Sumberrejo 0.812901   7.626947    578.0  10.658271
Bojonegoro - Kali Kethek 1.229414  11.687227    593.0  10.519296

=== 10 pos TERBAIK ===
             nama_pos     rmse  tma_range  n_baris  nrmse_pct
              Badegan 0.183938   2.057899    600.0   8.938133
                Napel 1.000615  11.230071    600.0   8.910143
                Jarum 

C:\Users\leouw\AppData\Local\Temp\ipykernel_26832\2980781329.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_pos = val_set.groupby('nama_pos').apply(


In [58]:
per_pos['total_se'] = per_pos['rmse']**2 * per_pos['n_baris']
per_pos['pct_of_total_se'] = per_pos['total_se'] / per_pos['total_se'].sum() * 100
per_pos_by_impact = per_pos.sort_values('total_se', ascending=False)

print('=== Ranking berdasar KONTRIBUSI ke skor leaderboard ===')
print(per_pos_by_impact[['nama_pos','rmse','n_baris','total_se','pct_of_total_se']].head(10).to_string(index=False))
print(f'\n10 pos teratas ini menyumbang: {per_pos_by_impact.head(10)["pct_of_total_se"].sum():.1f}% dari total error')

=== Ranking berdasar KONTRIBUSI ke skor leaderboard ===
                nama_pos     rmse  n_baris    total_se  pct_of_total_se
            Wonogiri Dam 1.944217    600.0 2267.987834        19.799313
       Boboh Kali Lamong 1.344880    600.0 1085.221719         9.473880
Bojonegoro - Kali Kethek 1.229414    593.0  896.294969         7.824568
       Floodway Bridge C 1.003799    600.0  604.566865         5.277810
                   Napel 1.000615    600.0  600.738763         5.244391
                Ketonggo 0.970850    600.0  565.529300         4.937016
            Karangnongko 0.967093    600.0  561.161541         4.898886
             Bengkelolor 0.954109    600.0  546.193832         4.768219
              Kedungupit 0.905646    600.0  492.117062         4.296134
                    Cepu 0.897172    598.0  481.341213         4.202062

10 pos teratas ini menyumbang: 70.7% dari total error


In [59]:
for in 

SyntaxError: invalid syntax (2463186271.py, line 1)

# Submission

In [ ]:
cat_final = CatBoostRegressor(**cat_params); cat_final.fit(train[feature_cols], train['tma_mdpl'])
xgb_final = XGBRegressor(**xgb_params); xgb_final.fit(train[feature_cols], train['tma_mdpl'])
lgbm_final = LGBMRegressor(**lgbm_params); lgbm_final.fit(train[feature_cols], train['tma_mdpl'])

KeyboardInterrupt: 

## XGB

In [ ]:
test_pred = xgb_final.predict(test[feature_cols])
submit_xgb = pd.DataFrame({'id': test['id'], 'tma_mdpl': test_pred})
submit_xgb.to_csv('outputs/submission_xgb.csv', index=False)

## LGBM

In [ ]:
test_pred = lgbm_final.predict(test[feature_cols])
submit_lgbm = pd.DataFrame({'id': test['id'], 'tma_mdpl': test_pred})
submit_lgbm.to_csv('outputs/submission_lgbm.csv', index=False)

## CatBoost

In [ ]:
test_pred = cat_final.predict(test[feature_cols])
submit_cat = pd.DataFrame({'id': test['id'], 'tma_mdpl': test_pred})
submit_cat.to_csv('outputs/submission_cat.csv', index=False)

## Ensemble Voting

In [ ]:
pred_cat_test = cat_final.predict(test[feature_cols])
pred_xgb_test = xgb_final.predict(test[feature_cols])
pred_lgbm_test = lgbm_final.predict(test[feature_cols])

final_pred = (best_weights[0]*pred_cat_test + best_weights[1]*pred_xgb_test + best_weights[2]*pred_lgbm_test)

submit_ensemble = pd.DataFrame({'id': test['id'], 'tma_mdpl': final_pred})
submit_ensemble.to_csv('outputs/submission_ensemble.csv', index=False)